In [26]:
from pymongo import MongoClient
from dotenv import load_dotenv
import os

load_dotenv()
collection_name = "articles_data"
mongo_uri = os.getenv("MONGO_URI_NAACP")
mongo_db_name = os.getenv("MONGO_DB_NAME_NAACP")
mongo_client = MongoClient(mongo_uri)

# Set up MongoDB connection
db = mongo_client[mongo_db_name]
collection = db[collection_name]


In [27]:
# Aggregation pipeline to find the most common coordinates with associated locations
pipeline = [
    {
        "$unwind": "$coordinates"  # Deconstructs the array into individual coordinates
    },
    {
        "$group": {
            "_id": "$coordinates",  # Group by coordinates
            "count": { "$sum": 1 },  # Count occurrences
        }
    },
    {
        "$sort": {
            "count": -1  # Sort by the count in descending order
        }
    }
]

# Execute the aggregation pipeline
common_coordinates = collection.aggregate(pipeline)

for entry in common_coordinates:
    print(f"Coordinates: {entry['_id']}")
    print(f"Count: {entry['count']}")


Coordinates: [-71.0588801, 42.3600825]
Count: 56
Coordinates: [-74.0059728, 40.7127753]
Count: 49
Coordinates: [-70.9078346, 42.3020647]
Count: 35
Coordinates: [-71.5828444, 42.5000919]
Count: 33
Coordinates: [-71.0898892, 42.339904]
Count: 31
Coordinates: [-71.1182488, 42.3744368]
Count: 23
Coordinates: [-71.068767, 42.36256789999999]
Count: 22
Coordinates: [-71.09416, 42.360091]
Count: 21
Coordinates: [-71.1218915, 42.3712317]
Count: 21
Coordinates: [-71.057716, 42.35807]
Count: 20
Coordinates: [-71.0734619, 42.3347657]
Count: 19
Coordinates: [-71.10727899999999, 42.3504017]
Count: 19
Coordinates: [-71.104215, 42.3356451]
Count: 19
Coordinates: [-71.0854325, 42.3284483]
Count: 18
Coordinates: [-70.8427794, 42.606693]
Count: 18
Coordinates: [-71.1053991, 42.3504997]
Count: 18
Coordinates: [-71.10245259999999, 42.3354472]
Count: 16
Coordinates: [-71.0972178, 42.3466764]
Count: 16
Coordinates: [-71.1192282, 42.3784622]
Count: 16
Coordinates: [-71.1068499, 42.3361779]
Count: 16
Coordinat

In [53]:
target_coordinate = [-71.1182488, 42.3744368]

documents = collection.find({"coordinates": target_coordinate})

In [54]:
matching_count = collection.count_documents({"coordinates": target_coordinate})
print(f"Number of matching documents: {matching_count}")

Number of matching documents: 23


In [50]:
locations_collection = db["locations_data"]

result = locations_collection.delete_many({"coordinates": target_coordinate})
print(f"Number of documents deleted: {result.deleted_count}")

Number of documents deleted: 1


In [35]:
for doc in documents:
    index = next((i for i, coord in enumerate(doc['coordinates']) if coord == target_coordinate), -1)
    
    if index > -1:
        new_coordinates = [coord for i, coord in enumerate(doc['coordinates']) if i != index]
        new_location = [loc for i, loc in enumerate(doc['locations']) if i != index]
        new_tract = [tract for i, tract in enumerate(doc['tracts']) if i != index]
        new_county = [county for i, county in enumerate(doc['counties']) if i != index]
        new_neighborhood = [neighborhood for i, neighborhood in enumerate(doc['neighborhoods']) if i != index]

        # Prepare the update operation
        update = {
            "$set": {
                "coordinates": new_coordinates,
                "locations": new_location,
                "tracts": new_tract,
                "counties": new_county,
                "neighborhoods": new_neighborhood
            }
        }
        
        # Apply the update operation
        collection.update_one({"_id": doc["_id"]}, update)
        print(f"Updated document with _id: {doc['_id']}")


print("Update operation completed.")

Updated document with _id: bb11e767fb932f8979cf47bcde892b79d833dab1c133ab21f7a4ca2ec30f5c7c
Updated document with _id: 5724a71705a0b5a4181bacd47506316f7b4063b58c626deeb90d4d3f91b8e463
Updated document with _id: 941f288d7e507414176263fdd7c7be03564b971149ab2c93fd0deb8b13fc5e27
Updated document with _id: 0d834791279bfbe4ec527c02eddf8b2478ccfe0ae8ceb4e48f3a54834673c4a3
Updated document with _id: 8352d4e9499d990e416a1fdeeab69243790dcf8d62ca3f66ad561f05a17f7422
Updated document with _id: ff5a5207d43d671db8d41e10b741e55a7884391728972ac96b55136500ead867
Updated document with _id: 26f713649a7827266d334d48e4024baf1e160d1b931aede225418569e306453f
Updated document with _id: f4a5517b585f5262b50a3d51e64a894d7a948893cd7baa33c8ea791d5fdf7e8f
Updated document with _id: 56c60211ddf13b1b01c3b18efd674ffe923aa98fa957049169af774bd58e23bf
Updated document with _id: 49539cb909d631e115aee64de75dc8b7287440cc8cf61d37a5894ff61697bd5c
Updated document with _id: ef3737b5f49c0d9532f5c36f38fcd25b6e4b4eeca91dcd917f34b

In [36]:
matching_count = collection.count_documents({"coordinates": target_coordinate})
print(f"Number of matching documents: {matching_count}")

Number of matching documents: 0
